In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, get_dataset_config_names
import random
import pandas as pd
from typing import Iterable

# ==============================================================================
# 🧠 [튜터 코멘트] 환영합니다, 예비 AI 엔지니어!
# ==============================================================================
# ✨ 오늘의 데이터셋: chkmie/uav-resilience-bench
# ✨ ✨ 의미: 이 데이터셋은 '드론'과 관련된 시간 흐름(Time-Series) 데이터예요.
# ✨ ✨ 데이터가 어려운 이유: 일반적인 드론 데이터는 날씨가 좋고 (Fair Weather)
# ✨ ✨ 전파 방해나 센서 오류 같은 '위험한 환경(Adversarial Condition)'을 시뮬레이션한 데이터라서,
# ✨ ✨ 얼마나 튼튼하게(Resilience) 드론이 길을 찾는지 테스트하는 데 쓰여요.
# ✨ ✨ 목표: GPS가 방해받을 때 (Jamming) 우리의 AI가 어떻게 위치를 추정할지 분석해보는 실습을 해볼 거예요!
# ==============================================================================

DATASET_ID = "chkmie/uav-resilience-bench"
SPLIT_NAME = "train"
SAMPLE_COUNT = 100 # 상위 100개 샘플만 사용하여 빠르게 동작합니다.

# 1. 데이터셋 로딩 전략 (스트리밍 vs. 로컬 다운로드)
print("🚀 데이터셋을 로딩하는 중입니다... 스트리밍 모드를 먼저 시도합니다.")
dataset = None
try:
    # 2. 스트리밍 모드 시도 (메모리 효율적)
    dataset = load_dataset(DATASET_ID, split=SPLIT_NAME, streaming=True)
    print("\n🟢 [성공] Streaming Mode로 데이터셋을 성공적으로 불러왔습니다! (메모리 절약!)")
except Exception as e:
    print(f"\n⚠️ [경고] Streaming 모드 로드 중 오류 발생: {e}")
    print("➡️ 로컬 다운로드 모드로 전환하여 상위 샘플을 불러옵니다.")
    try:
        # 스트리밍 실패 시, 작은 샘플만 다운로드하여 진행
        dataset = load_dataset(DATASET_ID, split=SPLIT_NAME, streaming=False)
        print("🟢 [성공] 일반 로딩 모드로 데이터를 불러왔습니다.")
    except Exception as e_fallback:
        print(f"❌ 데이터셋 로드에 실패했습니다. 오류: {e_fallback}")
        exit()

# 3. 샘플링 및 반복자 설정 (Constraints 준수)
# 스트리밍/비스트리밍 모두 .take() 메서드를 사용하여 상위 K개 샘플만 가져옵니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    print(f"\n⏳ 상위 {min(SAMPLE_COUNT, 100)}개 샘플만 추출합니다...")
    # 데이터셋의 모든 항목을 순회할 필요 없이 상위 K개만 Iterator로 만듭니다.
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 데이터셋 (Dataset)
    print("💡 Dataset 객체로 처리합니다.")
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)

# 4. 데이터 처리 및 분석 (핵심 로직)

# 실제 분석을 위해 상위 샘플들을 리스트로 변환하여 사용합니다.
sample_data_list = list(sampled_dataset_iterator)
print(f"✅ 총 {len(sample_data_list)}개의 샘플을 분석 대상으로 선정했습니다.")

# 결과를 저장할 리스트 초기화
jamming_samples = []
analysis_results = []

# 데이터 순회하며 전처리 및 특징 추출
for i, sample in enumerate(sample_data_list):
    # 필수 데이터 추출
    timestamp = sample['features']['timestamp_s']
    is_jammer_active = sample['features']['environment']['jammer_active']
    
    # 🚩 중요! Ground Truth 및 센서 추정값 추출 (리스트 형태이므로 첫 번째 요소 사용)
    pos_gt = sample['features']['ground_truth']['pos_ned_m'][0]
    ukf_pos = sample['features']['baseline']['ukf_pos_est'][0]
    
    # GPS 데이터 추출 및 유효성 확인
    is_gps_valid = sample['features']['sensors']['gps']['valid']
    
    # 🌟 오늘의 분석 주제: 방해(Jamming)가 발생했을 때 위치 추정 오류를 관찰합니다.
    if is_jammer_active:
        jamming_samples.append({
            'timestamp': timestamp,
            'pos_gt': pos_gt,
            'ukf_pos': ukf_pos,
            'gps_valid': is_gps_valid,
            'jamming': True
        })
    else:
        jamming_samples.append({
            'timestamp': timestamp,
            'pos_gt': pos_gt,
            'ukf_pos': ukf_pos,
            'gps_valid': is_gps_valid,
            'jamming': False
        })

# 결과를 Pandas DataFrame으로 변환하여 쉬운 분석 환경 만듭니다.
df_analysis = pd.DataFrame(jamming_samples)

# 5. 정량적 분석 및 시각화

print("\n" + "="*70)
print("✨ 🛰️ 분석 결과: GPS 교란 환경에서의 자율항법 성능 분석 ✨")
print("="*70)

# 5-1. Jamming 조건에서의 성능 지표 분석
jammed_df = df_analysis[df_analysis['jamming'] == True].copy()
clean_df = df_analysis[df_analysis['jamming'] == False].copy()

if not jammed_df.empty:
    # 1. 위치 오차 계산 (L2 Norm: sqrt(dx^2 + dy^2 + dz^2))
    jammed_df['error'] = np.sqrt(
        (jammed_df['pos_gt'][:, 0] - jammed_df['ukf_pos'][:, 0])**2 +
        (jammed_df['pos_gt'][:, 1] - jammed_df['ukf_pos'][:, 1])**2 +
        (jammed_df['pos_gt'][:, 2] - jammed_df['ukf_pos'][:, 2])**2
    )
    
    # 2. Jamming 시 평균 오류와 GPS 유효성 비교
    avg_error_jammed = jammed_df['error'].mean()
    avg_error_clean = clean_df['error'].mean()
    
    print("\n[📊 1. 위치 추정 오차 (Position Error) 비교]")
    print("-" * 30)
    print(f"✅ 정상 환경 평균 오차 (Clean): {avg_error_clean:,.2f} 미터")
    print(f"🚨 교란 환경 평균 오차 (Jammed): {avg_error_jammed:,.2f} 미터")
    
    if avg_error_jammed > avg_error_clean * 1.5:
        print("🌟 분석 결과: 교란 환경에서 오차가 눈에 띄게 증가했습니다! (예상대로)")
    else:
        print("💡 분석 결과: 오차 변화가 미미합니다. 다른 요소를 살펴봐야겠어요.")
    
    print("\n[🚦 2. GPS 센서 신뢰성 분석]")
    # Jamming이 활성화되었을 때, GPS 유효성(gps_valid)이 어떻게 변하는지 확인
    invalid_count_jamming = jammed_df[jammed_df['gps_valid'] == False].shape[0]
    total_jamming = jammed_df.shape[0]
    
    print(f"   - 총 교란 샘플 수: {total_jamming}개")
    print(f"   - 교란 시 GPS 무효화된 샘플 수: {invalid_count_jamming}개 (비율: {(invalid_count_jamming/total_jamming)*100:.1f}%)")
    print("🏆 Conclusion: GPS가 끊길 때(False), AI는 IMU나 UKF 필터 같은 다른 방법을 써야 해요! (진정한 Resilience!)")

# 6. 간단한 시각화 (Matplotlib)
print("\n[🎨 3. 시간 경과에 따른 위치 변화 시각화]")
plt.figure(figsize=(12, 6))

# 3D 플롯: 시간(x), Y좌표(y), Z좌표(z)
scatter_clean = clean_df[clean_df['jamming'] == False]
scatter_jammed = jammed_df[jammed_df['jamming'] == True]

# Ground Truth (진짜 위치)를 파란색으로 연결
plt.plot(
    [s['timestamp'] for s in df_analysis], 
    [s['pos_gt'][1] for s in df_analysis], 
    label='Ground Truth (True Path)', color='blue', linewidth=1
)

# UKF 추정치 (AI가 생각한 위치)를 점으로 표시하여 비교
plt.scatter(
    [s['timestamp'] for s in df_analysis], 
    [s['ukf_pos'][1] for s in df_analysis], 
    c = np.where(np.array([s['jamming'] for s in df_analysis]), 
                     'red' if np.array([s['jamming'] for s in df_analysis]) else 'green',
                     'green'), 
    label='UKF Estimate (AI Prediction)', s=5
)

plt.title("UAV Position Tracking: Clean vs. Jammed Environment", fontsize=14)
plt.xlabel("Time (seconds)")
plt.ylabel("Y Position (meters)")
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()

print("✨ 모든 실습이 완료되었습니다! 전파 방해 환경에서도 AI가 데이터를 분석하고 경고하는 과정을 성공적으로 묘사했습니다.")